#### Стеганография аудио и видео информации. Лабораторная работа №3
|   Группа          |   ФИО             |   
|   :------------:  |   :------------:  |
|   М092501(71)     |   Шарибжанов И.Т. |

**Цель работы:**
Изучение метода скрытия данных в аудиосигналах методом расширения спектра, получение навыков реализация изученного метода в программной среде

##### Импорт библиотек и генерация аудиоконтейнера

In [ ]:
import numpy as np
from scipy.io import wavfile

# Функция для создания тестового аудиоконтейнера (B.wav)
def create_dummy_audio(filename, duration=3.0, sample_rate=44100):
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    # Генерируем простой тон 440 Гц
    audio = np.sin(440 * 2 * np.pi * t)
    # Конвертируем в формат 16-bit PCM
    audio_int16 = np.int16(audio * 32767)
    wavfile.write(filename, sample_rate, audio_int16)
    print(f"Создан тестовый контейнер: {filename}")

create_dummy_audio('B.wav')

# Создаем файл с сообщением
my_surname = "Шарибжанов"
with open('A.txt', 'w', encoding='utf-8') as f:
    f.write(my_surname)
print(f"Создан файл с секретным сообщением: A.txt")

Создан тестовый контейнер: B.wav
Создан файл с секретным сообщением: A.txt


##### Вспомогательные функции

In [ ]:
def text_to_bits(text):
    """Перевод строки (UTF-8) в биты."""
    bytes_data = text.encode('utf-8')
    return ''.join(format(b, '08b') for b in bytes_data)

def bits_to_text(bits):
    """Перевод битов обратно в строку (UTF-8)."""
    byte_array = bytearray(int(bits[i:i+8], 2) for i in range(0, len(bits), 8))
    return byte_array.decode('utf-8', errors='ignore')

def generate_psp(N, seed=42):
    """Генератор ПСП (псевдослучайной последовательности)."""
    np.random.seed(seed) # Фиксируем seed, чтобы ключ совпадал при дешифровке
    # Генерируем массив из -1 и 1
    return np.random.choice([-1.0, 1.0], size=N)

##### Функция встраивания (Стегакодер)

In [ ]:
def embed_audio(msg_file, container_file, stego_file):
    with open(msg_file, 'r', encoding='utf-8') as f:
        message = f.read()
        
    bin_data = text_to_bits(message)
    num_bits = len(bin_data)
    
    sample_rate, audio_data = wavfile.read(container_file)
    is_stereo = (audio_data.ndim == 2)
    
    # НОРМАЛИЗАЦИЯ: переводим в диапазон [-1.0, 1.0]
    audio = audio_data.astype(np.float64) / 32768.0
    
    # Вычисляем, сколько звуковых отсчетов приходится на скрытие одного бита информации
    num_samples = len(audio)
    N = num_samples // num_bits
    
    if N <= 0:
        print("Ошибка: Недостаточно большой контейнер")
        return False
        
    print(f"Длина аудио: {num_samples}. Бит: {num_bits}. Отсчетов на бит: {N}")
    
    psp = generate_psp(N)
    stego_audio = np.copy(audio)
    
    # Встраивание по формулам методички
    for i in range(num_bits):
        bit = bin_data[i]
        
        # Шаг 1 из методички: если бит 0, инвертируем ПСП
        if bit == '0':
            pspmes = -psp * 0.0005
        else:
            pspmes = psp * 0.0005
            
        start = i * N
        end = (i + 1) * N
        Y = audio[start:end]
        
        # Шаг 3 из методички: Y + pspmes * (Y + 2)
        if is_stereo:
            # Расширяем массив pspmes для двух каналов
            stego_audio[start:end, 0] = Y[:, 0] + pspmes * (Y[:, 0] + 2.0)
            stego_audio[start:end, 1] = Y[:, 1] + pspmes * (Y[:, 1] + 2.0)
        else:
            stego_audio[start:end] = Y + pspmes * (Y + 2.0)
            
    # ДЕНОРМАЛИЗАЦИЯ И ЗАЩИТА
    # Возвращаем в диапазон [-32768, 32767]
    stego_audio_scaled = stego_audio * 32768.0
    
    # Жестко обрезаем пики, чтобы избежать переполнения
    stego_audio_clipped = np.clip(stego_audio_scaled, -32768, 32767)
    
    wavfile.write(stego_file, sample_rate, stego_audio_clipped.astype(np.int16))
    print(f"Работа завершена. Стеганоконтейнер сохранен в {stego_file}")
    return num_bits

##### Функция извлечения (Стегадекодер)

In [ ]:
def extract_audio(container_file, stego_file, output_file, num_bits):
    _, y_data = wavfile.read(container_file)
    _, x_data = wavfile.read(stego_file)
    is_stereo = (y_data.ndim == 2)
    
    # НОРМАЛИЗАЦИЯ: переводим в диапазон [-1.0, 1.0]
    orig_audio = y_data.astype(np.float64) / 32768.0
    mod_audio = x_data.astype(np.float64) / 32768.0
    
    N = len(orig_audio) // num_bits
    psp = generate_psp(N)
    
    extracted_bits = ""
    
    for i in range(num_bits):
        start = i * N
        end = (i + 1) * N
        
        Y = orig_audio[start:end]
        X = mod_audio[start:end]
        
        # Формула из DeST: (X - Y) / (Y + 2)
        if is_stereo:
            Res_left = (X[:, 0] - Y[:, 0]) / (Y[:, 0] + 2.0)
            Res_right = (X[:, 1] - Y[:, 1]) / (Y[:, 1] + 2.0)
            # Суммируем корреляцию с двух каналов
            correlation = np.dot(Res_left, psp) + np.dot(Res_right, psp)
        else:
            Res = (X - Y) / (Y + 2.0)
            correlation = np.dot(Res, psp)
            
        # Если корреляция больше нуля, значит ПСП совпадала (не инвертирована) -> бит 1
        if correlation > 0:
            extracted_bits += '1'
        else:
            extracted_bits += '0'
            
    secret_message = bits_to_text(extracted_bits)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(secret_message)
        
    print(f"Извлеченное сообщение: {secret_message}")

##### Запуск и выполнение программы

In [ ]:
print("--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---")
# Встраиваем данные из A.txt в B.wav и сохраняем как C.wav
hidden_bits_count = embed_audio('A.txt', 'B.wav', 'C.wav')

print("\n--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---")
# Извлекаем данные из C.wav и сохраняем в decrypted_A.txt
if hidden_bits_count:
    extract_audio('B.wav', 'C.wav', 'decrypted_A.txt', hidden_bits_count)

--- ЭТАП 1: СКРЫТИЕ ДАННЫХ ---
Длина аудио: 132300. Бит: 160. Отсчетов на бит: 826
Работа завершена. Стеганоконтейнер сохранен в C.wav

--- ЭТАП 2: ИЗВЛЕЧЕНИЕ ДАННЫХ ---
Извлеченное сообщение: Шарибжанов
